# Over-engineering Connect 4

## Motivations

In any field within quantum mechanics, we find that symmetries are often used to make our lives easier and simpler, and they are often unavoidable. Indeed, when Emmy Noether published her theorem in 1918 indicating that our familiar laws are often DEFINED by the symmetries they adhere to, they really are both something to rely on, and something to exploit as often as possible.

Connect 4 is a game where one tries to Connect 4 of the same coloured piece in a grid. It is contested between two players which each have their own colour. Due to it's simple nature, as you may expect, we have already solved the game. Whoever goes first can always force a win, and high level strategies involve just putting as many pieces as you can down the central column and going from there. So you may wonder, how can one overengineer this completed problem to a ridiculous degree, and why were the words 'quantum mechanics' mentioned in the first line. Well we can in fact use quantum machine learning (QML) and exploit the natural symmetries of this game to optimize our solution!

### The setup

Due to budget cuts, I'll be slashing the standard board down to a 5x4 grid, but the problem is scalable. In fact, the exploitation of symmetries only works better the larger the board, so feel free to follow along and make the grid as large as you want - sadly my laptop would explode if I went any larger than this.

Let's get the boring coding out the way first:

In [6]:
import numpy as np
import random

COLUMNS = 5
ROWS = 4

def create_board():
    return np.zeros((ROWS, COLUMNS))

def possible_moves(board):
    # Return the columns where a piece can be dropped
    moves = []
    rows, cols = board.shape
    for col in range(cols):
        if board[0, col] == 0:
            moves.append(col)
    return moves

def drop_piece(board, col, player):
    # Drop a player's piece into the chosen column
    rows, cols = board.shape
    # Bottom row up to the top
    for row in range(rows - 1, -1, -1):
        # If the space is free, drop the piece
        if board[row, col] == 0:
            board[row, col] = player
            return board
        
def four_in_a_row(board, row, col, d_row, d_col, player):
    # Check for 4 of a player's piece starting at (row, col)
    rows, cols = board.shape
    for step in range(4):
        r = row + step * d_row
        c = col + step * d_col
        # Check for board's boundaries
        if not (0 <= r < rows and 0 <= c < cols):
            return False
        if board[r, c] != player:
            return False
    return True
        
def check_win(board, player):
    # Return True if a 4 in a row anywhere is found
    # Assign directions of right, down, and the diagonals
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]

    rows, cols = board.shape
    for row in range(rows):
        for col in range(cols):
            for d_row, d_col in directions:
                if four_in_a_row(board, row, col, d_row, d_col, player):
                    return True
    return False

def random_place(board, player):
    col = random.choice(possible_moves(board))
    return drop_piece(board, col, player)

def evaluate_game(board):
    for player in [-1, 1]:
        if check_win(board, player):
            return player
    if len(possible_moves(board)) == 0:
        return 0 # Draw
    return None

def play_game():
    board, winner = create_board(), None
    while winner is None:
        for player in [1, -1]:
            board = random_place(board, player)
            winner = evaluate_game(board)
            if winner is not None:
                break
    # Flatten the board to use for training
    return [board.flatten(), winner]

A few notes on the game I have devised. Firstly, the game immediately ends when a winner is found. Secondly, the pieces are placed randomly - but this is just to create our dataset that we can learn from. 

## Quantumizing

So we've done the so-called 'classical' part, now to get silly. 

### Defining our model

I will be defining the problem as follows. Each cell in the connect 4 board will be represented as a qubit. I will label each cell going from the top right, going across the row, then moving to the next column. This makes for 20 qubits, labelled from 0 to 19, all in a line, ready to achieve great things together.

But what of the symmetries of the model? Well, looking at our 5x4 board, we can immediately see that we have a point of reflection down the central column. So flipping the board round this axis will yield the same board - or if two players were playing the game, simply turning the board doesn't change the state of the game. Using this idea, we can see where points of symmetry DON'T lie. Let's say one of the players flips the board on its head. Pieces will go everywhere and the game will be ruined, so the existence of gravity actually cuts down our potential symmetries, and we are left with just the one.

There is another symmetry that is not immediately obvious, as it is a different type of symmetry. This would be doing a 'colour-swap' so to say, the yellow pieces to the red pieces. Now, this would of course change the outcome of the game, even though the number of pieces in each column has not changed. This is the idea of equivariance, as opposed to our invariant reflection symmetry. 

### Group Theory

To keep it brief, let's only think about the reflection symmetry. This is a Z2 symmetry. Turns out we can continue to represent this whole problem as a group. Each 